# 🚀 真·DINOv3 特征预提取流水线 (VPT → HDF5 → HuggingFace)

把 VPT/BASALT 的 `.mp4` 帧逐帧过**真实 DINOv3**,把池化嵌入预计算并上传到 HF,供 RSSM 世界模型离线消费(训练时不再跑 DINO)。

**严格复用仓库代码,无任何 mock**:
1. 模型 = `net.backbone.build_backbone(BackboneConfig(kind="dinov3"))` → `facebook/dinov3-vits16-pretrain-lvd1689m`(ViT-S/16,**384 维**,patch16,4 register,gated)
2. 特征 = `train_rssm.FrozenBackbonePerception` 契约:ImageNet 归一化 → 丢 CLS/register → **patch 均值池化 → [N, 384]**
3. 数据 = `domains.minecraft.vpt_dataset` 的 `.mp4`+`.jsonl` 解码;动作走同一个 `_action_vec`
4. 多线程异步 HDF5(GZIP 分块)+ HF Hub 后台上传 + 硬件实时仪表盘

**每帧三路对齐落盘**:`dino_features [N,384]` + `actions [N,22]`(同 `vpt_action` 布局)+ `clip_id [N]`(全局单调;下游按文件名拼接所有 part_*.h5 后按 clip_id 分组,即得逐 clip 的有序帧序列 + 对齐动作)。文件级 attrs 记录模型/维度/分辨率/camera_scale/动作布局。

**前置**:① GPU 运行时;② 在 HF 网页接受 DINOv3 许可证;③ 配置 `HF_TOKEN`(Colab 🔑 面板,需 **write** 权限)且对 gated 权重有访问权。

In [ ]:
# ===== 1. 安装依赖 + 克隆仓库 + 解析 HF Token =====
# h5py/psutil 不在仓库 requirements 里(本笔记本的特征缓存与监控用),单独装;
# transformers 升级以确保支持 DINOv3 架构。
!pip install -q -U transformers
!pip install -q h5py psutil huggingface_hub opencv-python pyyaml

import os, sys, time, queue, threading, subprocess
import numpy as np
import h5py
import psutil
import torch
from torch.amp import autocast            # 新写法;torch.cuda.amp.autocast 已废弃
from huggingface_hub import HfApi

# 克隆本项目仓库(已存在则跳过),并把仓库根加入 import 路径,后续直接用仓库里的真实模块
REPO_URL = "https://github.com/OopsYouDiedE/tao-not-42-base-refactor-world-model-contract.git"
REPO_DIR = "/content/tao-not-42-base-refactor-world-model-contract"
if not os.path.isdir(os.path.join(REPO_DIR, "net")):
    !git clone {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# 复用仓库的 token 解析(环境变量 → Colab Secret → .env → 缓存登录),并写回 os.environ 供 HF 自动鉴权
from utils.io import get_hf_token
tok = get_hf_token()
print(f"✅ 仓库就绪 @ {REPO_DIR}")
print(f"{'✅' if tok else '⚠️'} HF token: {'已解析(后续 HF 调用自动鉴权)' if tok else '未找到 —— DINOv3 是 gated,下载会失败;请配置 Colab Secret HF_TOKEN'}")


In [ ]:
# ===== 2. 异步 HDF5 写入器与上传器(同存 features + actions + clip_id)=====
def hf_upload(filepath, repo_id="your-username/dino-minecraft-features"):
    size_gb = os.path.getsize(filepath) / (1024 ** 3)
    print(f"\n[Uploader] 🚀 异步上传: {filepath} ({size_gb:.2f} GB)")
    try:
        api = HfApi()  # token 由 cell 1 的 get_hf_token() 写入 os.environ,自动鉴权
        filename = os.path.basename(filepath)
        # 如果仓库不存在，则自动创建一个 Dataset 仓库（exist_ok=True 已存在也不报错;需 write 权限）
        api.create_repo(repo_id=repo_id, repo_type="dataset", exist_ok=True)
        api.upload_file(
            path_or_fileobj=filepath,
            path_in_repo=f"data/{filename}",
            repo_id=repo_id,
            repo_type="dataset",
        )
        print(f"[Uploader] ✅ 上传完成: {filepath}")
        # 上传完成后清理本地磁盘，极致省流
        os.remove(filepath) 
    except Exception as e:
        print(f"[Uploader] ❌ 上传失败 {filepath}: {e}")

class AsyncHDF5Writer:
    """逐帧 (dino_features[E] + actions[A] + clip_id) 三路对齐落盘。

    clip_id 是**全局单调递增**的整数:下游按文件名顺序拼接所有 part_*.h5,再按 clip_id
    分组即可还原"逐 clip 的有序帧序列 + 对齐动作"(一段 clip 跨文件也安全,因 id 连续)。
    每个文件写入溯源 attrs(模型/维度/分辨率/camera_scale/动作布局)。
    """
    _STOP = object()  # 上传队列的关闭哨兵

    def __init__(self, output_dir="/content/hdf5_cache", max_file_bytes=50*(1024**2),
                 upload_fn=None, emb_dim=384, act_dim=22, attrs=None):
        self.output_dir = output_dir
        self.max_file_bytes = max_file_bytes
        self.upload_fn = upload_fn
        self.emb_dim = emb_dim          # 嵌入维 = 骨干 enc_dim(DINOv3 ViT-S/16 = 384)
        self.act_dim = act_dim          # 动作维 = vpt_action.ACTION_DIM(22)
        self.attrs = dict(attrs or {})  # 溯源元信息(写进每个文件的 HDF5 attrs)
        os.makedirs(output_dir, exist_ok=True)
        self.memory_buffer, self.buffer_lock = [], threading.Lock()  # list[(feat,act,cid)]
        self.file_index = 0
        self.current_h5, self.current_filepath = None, None
        self.upload_queue = queue.Queue()
        self._open_new_file()
        self.running = True
        
        self.writer_thread = threading.Thread(target=self._writer_loop, daemon=True)
        self.uploader_thread = threading.Thread(target=self._uploader_loop, daemon=True)
        self.writer_thread.start()
        self.uploader_thread.start()

    def _open_new_file(self):
        if self.current_h5 is not None: self.current_h5.close()
        self.current_filepath = os.path.join(self.output_dir, f"dino_part_{self.file_index:04d}.h5")
        self.current_h5 = h5py.File(self.current_filepath, 'w')
        self.ds_feat = self.current_h5.create_dataset(
            'dino_features', shape=(0, self.emb_dim), maxshape=(None, self.emb_dim), dtype='float32',
            chunks=(2048, self.emb_dim), compression="gzip", compression_opts=4)
        self.ds_act = self.current_h5.create_dataset(
            'actions', shape=(0, self.act_dim), maxshape=(None, self.act_dim), dtype='float32',
            chunks=(2048, self.act_dim), compression="gzip", compression_opts=4)
        self.ds_cid = self.current_h5.create_dataset(
            'clip_id', shape=(0,), maxshape=(None,), dtype='int32',
            chunks=(8192,), compression="gzip", compression_opts=4)
        for k, v in self.attrs.items():
            self.current_h5.attrs[k] = v
        self.file_index += 1

    def append_data(self, feats, actions, clip_ids):
        """三路逐帧对齐:feats[b,E] f32 / actions[b,A] f32 / clip_ids[b] int32。"""
        with self.buffer_lock:
            self.memory_buffer.append((feats, actions, clip_ids))

    def _writer_loop(self):
        # 退出条件加上 memory_buffer：running 置 False 后仍把缓冲排空，避免丢尾部数据
        while self.running or self.memory_buffer:
            time.sleep(0.5)
            with self.buffer_lock:
                data_to_write, self.memory_buffer = self.memory_buffer, []
            if data_to_write:
                feats = np.concatenate([d[0] for d in data_to_write], axis=0)
                acts  = np.concatenate([d[1] for d in data_to_write], axis=0)
                cids  = np.concatenate([d[2] for d in data_to_write], axis=0)
                n0 = self.ds_feat.shape[0]
                for ds, arr in ((self.ds_feat, feats), (self.ds_act, acts), (self.ds_cid, cids)):
                    ds.resize(n0 + arr.shape[0], axis=0)
                    ds[n0:] = arr
                self.current_h5.flush()
                if os.path.getsize(self.current_filepath) >= self.max_file_bytes:
                    self.current_h5.close()
                    self.upload_queue.put(self.current_filepath)
                    self.current_h5 = None
                    self._open_new_file()

    def _uploader_loop(self):
        # 阻塞式取队列，收到哨兵才退出 —— 保证关闭前队列里所有文件（含最后一个分片）都被处理
        while True:
            filepath = self.upload_queue.get()
            try:
                if filepath is self._STOP:
                    break
                if self.upload_fn: self.upload_fn(filepath)
            finally:
                self.upload_queue.task_done()

    def close(self):
        self.running = False
        print("[System] 🛑 等待后台落盘与队列中剩余文件上传完毕...")
        # 1) 先等写线程把缓冲排空并停止（停止后不会再往队列里 put）
        self.writer_thread.join()
        # 2) 把最后一个未满分片也排队上传（否则尾部文件永远不会上传）
        if self.current_h5 is not None:
            self.current_h5.close()
            self.current_h5 = None
            self.upload_queue.put(self.current_filepath)
        # 3) 放入哨兵，上传线程处理完队列里全部文件后才退出
        self.upload_queue.put(self._STOP)
        self.uploader_thread.join()
        print("[System] ✅ 安全退出完成！")


In [ ]:
# ===== 3. 真·DINOv3 特征提取(复用仓库 net/backbone + FrozenBackbonePerception 契约)=====
import json
import contextlib
import cv2
import torch.nn.functional as F
from net.config import BackboneConfig
from net.backbone import build_backbone
# 仓库里的真实数据接口:(mp4,jsonl) 配对器 + 单帧动作编码(与 vpt_dataset._load_clip 同一函数)
from domains.minecraft.vpt_dataset import _pair_list, _action_vec
from domains.minecraft.vpt_action import CAMERA_SCALE, ACTION_DIM

# ---- 配置 ----
REPO_ID  = "unjustify/composed_VPT_dataset"   # 笔误已修正(dataset);上传目标 HF dataset 仓库
DATA_DIR = "/content/vpt_data"                # 放 VPT 的 .mp4+.jsonl(或先跑 tools/download_sample_data.py)
IMG_SIZE = 128                                # 本项目 Minecraft 帧分辨率(patch16 → 8×8 grid)
BATCH    = 256                                # 单次过 DINOv3 的帧数(按显存调)

dev = "cuda" if torch.cuda.is_available() else "cpu"
if dev == "cpu":
    print("⚠️ 未检测到 GPU,将用 CPU 跑 DINOv3(很慢);建议切到 GPU 运行时。")

# ---- 加载真 DINOv3(仓库默认:facebook/dinov3-vits16-pretrain-lvd1689m,ViT-S/16,384维,gated)----
backbone, patch, enc_dim, n_reg, kind = build_backbone(BackboneConfig(kind="dinov3"))
backbone = backbone.to(dev).eval()
for p in backbone.parameters():
    p.requires_grad_(False)
model_repo = getattr(backbone.config, "_name_or_path", kind)
print(f"✅ DINOv3 loaded: {model_repo} | enc_dim={enc_dim} patch={patch} n_register={n_reg} | device={dev}")

# ImageNet 归一化常数(与 net/world_model.py / train_rssm.FrozenBackbonePerception 完全一致)
_MEAN = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1)
_STD  = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)

@torch.no_grad()
def encode_frames(img):
    """[B,3,H,W] float∈[0,1] → [B,enc_dim] 逐帧池化嵌入。
    复刻 FrozenBackbonePerception.forward:整除补齐 + ImageNet 归一化 + 丢 CLS/register +
    patch 均值池化。这正是 RSSM 世界模型消费的 grounding 目标(emb_dim=384)。"""
    H, W = img.shape[-2:]
    ps = patch
    H2, W2 = max(ps, (H // ps) * ps), max(ps, (W // ps) * ps)
    if (H2, W2) != (H, W):
        img = F.interpolate(img, size=(H2, W2), mode="bilinear", align_corners=False)
    img = (img - _MEAN) / _STD
    tokens = backbone(pixel_values=img).last_hidden_state[:, 1 + n_reg:, :]
    return tokens.float().mean(dim=1)

def decode_clip(mp4_path, jsonl_path, img_size, camera_scale):
    """解码一段 mp4+jsonl → (img uint8[n,3,H,W] RGB, act f32[n,ACTION_DIM])。
    复刻 vpt_dataset._load_clip:帧 resize+RGB,动作走同一个 _action_vec,按 n=min 对齐。"""
    cap = cv2.VideoCapture(mp4_path)
    frames = []
    while True:
        ret, f = cap.read()
        if not ret:
            break
        if img_size:
            f = cv2.resize(f, (img_size, img_size), interpolation=cv2.INTER_AREA)
        frames.append(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
    cap.release()
    if not frames:
        return None
    acts = []
    with open(jsonl_path, "r", encoding="utf-8") as fh:
        for line in fh:
            acts.append(_action_vec(json.loads(line), camera_scale))
    n = min(len(frames), len(acts))
    if n == 0:
        return None
    img = torch.from_numpy(np.stack(frames[:n])).permute(0, 3, 1, 2).contiguous()  # uint8 [n,3,H,W]
    act = torch.stack(acts[:n]).numpy().astype(np.float32)                          # [n,ACTION_DIM]
    return img, act

# ---- 提取主循环 ----
upload = lambda p: hf_upload(p, repo_id=REPO_ID)
provenance = {
    "model_repo": str(model_repo), "emb_dim": int(enc_dim), "act_dim": int(ACTION_DIM),
    "img_size": int(IMG_SIZE), "camera_scale": float(CAMERA_SCALE),
    "action_layout": "mouse_dx,mouse_dy + 20 keys (domains.minecraft.vpt_action.VPT_KEYS)",
    "pooling": "drop CLS+register, mean over patch tokens (FrozenBackbonePerception)",
}
# emb_dim/act_dim 跟随真实契约;1GB 切片一次上传(按需调阈值)
writer = AsyncHDF5Writer(max_file_bytes=1024*1024*1024, upload_fn=upload,
                         emb_dim=enc_dim, act_dim=ACTION_DIM, attrs=provenance)

pairs = _pair_list(DATA_DIR)
assert pairs, f"❌ {DATA_DIR} 里没有成对的 .mp4/.jsonl —— 先放数据或跑 tools/download_sample_data.py"
print(f"🚀 开始提取 DINOv3 特征:{len(pairs)} 段视频 → HF dataset {REPO_ID}")

psutil.cpu_percent()
last_net, last_time, processed_frames, step = psutil.net_io_counters(), time.time(), 0, 0
amp_ctx = (lambda: autocast('cuda', dtype=torch.float16)) if dev == "cuda" else contextlib.nullcontext

try:
    for clip_id, (mp4, jsonl) in enumerate(pairs):
        out = decode_clip(mp4, jsonl, IMG_SIZE, CAMERA_SCALE)
        if out is None:
            print(f"⚠️ 跳过空/坏视频: {mp4}")
            continue
        img_u8, act = out                                            # uint8 [n,3,128,128] / f32 [n,22]
        n = img_u8.shape[0]
        for i in range(0, n, BATCH):
            chunk = img_u8[i:i+BATCH].to(dev).float() / 255.0        # [b,3,128,128] ∈ [0,1]
            with amp_ctx():
                emb = encode_frames(chunk)                           # [b, enc_dim=384]
            b = chunk.shape[0]
            cids = np.full(b, clip_id, dtype=np.int32)
            writer.append_data(emb.cpu().numpy().astype(np.float32), act[i:i+b], cids)
            processed_frames += b
            step += 1

            # --- 🕵️ 硬件级监控仪 ---
            if step % 10 == 0:
                c_time, c_net = time.time(), psutil.net_io_counters()
                dt = max(c_time - last_time, 1e-6)
                up_mb = (c_net.bytes_sent - last_net.bytes_sent) / dt / 1024 / 1024
                cpu_u = psutil.cpu_percent()
                try:
                    gpu_u = subprocess.check_output(
                        ["nvidia-smi", "--query-gpu=utilization.gpu", "--format=csv,noheader,nounits"],
                        text=True).strip().split('\n')[0]
                except Exception:
                    gpu_u = "N/A"
                gpu_mem = torch.cuda.memory_allocated() / 1024**3 if torch.cuda.is_available() else 0.0
                print(f"[Step {step:04d}] clip {clip_id+1}/{len(pairs)} | 🖥️ CPU: {cpu_u:04.1f}% | "
                      f"🎮 GPU: {gpu_u:>3}% (显存 {gpu_mem:.1f}G) | ⚡ {processed_frames/dt:.1f} 帧/秒 | "
                      f"⬆️ {up_mb:5.1f} MB/s | 🧠 缓冲 {len(writer.memory_buffer):>2} 批")
                last_time, last_net, processed_frames = c_time, c_net, 0
finally:
    writer.close()
